# Downloading SDO/AIA Data

This notebook is a reusable tutorial for downloading SDO/AIA images with SunPy.

Use it when you need new AIA data and want a reminder of the two common routes:

1. **Normal SunPy/Fido search and download**: usually the quickest and easiest option.
2. **JSOC export through SunPy**: usually more stable for AIA cutouts or well-defined JSOC queries, but slower because the data are staged before download.

The main things to change next time are in the parameter cell below: time range, wavelength, cadence, output folder, and JSOC email.

## 0. Setup

Install the usual solar data tools if they are missing:

```bash
pip install sunpy astropy drms parfive
```

You only need `drms` for the JSOC route. The normal route works through SunPy's `Fido` interface.

In [ ]:
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import sunpy.map
from sunpy.net import Fido, attrs as a

## 1. Change These Parameters Next Time

This is the main cell to edit when you come back for a different dataset.

- `START_TIME` and `END_TIME`: UTC time range.
- `WAVELENGTH`: AIA channel, for example `94`, `131`, `171`, `193`, `211`, `304`, `335`, `1600`, or `1700` Angstrom.
- `SAMPLE_CADENCE`: how often to sample files from the full AIA cadence. Use a larger cadence while testing.
- `JSOC_NOTIFY_EMAIL`: required by JSOC exports. Use the email registered with JSOC, or your usual notification email.

In [ ]:
START_TIME = "2014-01-01 00:00:00"
END_TIME = "2014-01-01 00:10:00"

WAVELENGTH = 171 * u.angstrom
SAMPLE_CADENCE = 2 * u.minute

BASE_DOWNLOAD_DIR = Path("data/aia_downloads")
NORMAL_DOWNLOAD_DIR = BASE_DOWNLOAD_DIR / "normal_fido"
JSOC_DOWNLOAD_DIR = BASE_DOWNLOAD_DIR / "jsoc"

JSOC_NOTIFY_EMAIL = "toshuho@gmail.com"

NORMAL_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
JSOC_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

## 2. Normal Method: SunPy Fido

Start here first. This is the simplest route and is often fast enough for small data requests.

This search asks for AIA files matching your time range, wavelength, and sampling cadence. It lets SunPy choose an available data provider.

In [ ]:
normal_query = Fido.search(
    a.Time(START_TIME, END_TIME),
    a.Instrument.aia,
    a.Wavelength(WAVELENGTH),
    a.Sample(SAMPLE_CADENCE),
)

normal_query

### Download The Normal Search Results

Run this after checking that the search results look right. For large requests, test with a short time range first.

In [ ]:
normal_files = Fido.fetch(
    normal_query,
    path=str(NORMAL_DOWNLOAD_DIR / "{file}"),
)

normal_files

### Quick Preview

Open the first downloaded FITS file as a SunPy map to check that you have the right data.

In [ ]:
if len(normal_files) > 0:
    aia_map = sunpy.map.Map(normal_files[0])

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(projection=aia_map)
    aia_map.plot(axes=ax)
    aia_map.draw_limb(axes=ax)
    plt.show()
else:
    print("No files were downloaded. Check the search result above.")

## 3. JSOC Method: More Stable, Slower

Use JSOC when the normal route is unreliable, when you need a more precise AIA query, or when you want to request cutouts.

JSOC is slower because it stages an export request first. Your `Notify` email is required so JSOC can associate the export with you.

In [ ]:
jsoc_query = Fido.search(
    a.Time(START_TIME, END_TIME),
    a.jsoc.Series("aia.lev1_euv_12s"),
    a.jsoc.Segment("image"),
    a.Wavelength(WAVELENGTH),
    a.Sample(SAMPLE_CADENCE),
    a.jsoc.Notify(JSOC_NOTIFY_EMAIL),
)

jsoc_query

### Download The JSOC Export

This can take longer than the normal route. If it appears to pause, it is usually waiting for JSOC to prepare the export.

In [ ]:
jsoc_files = Fido.fetch(
    jsoc_query,
    path=str(JSOC_DOWNLOAD_DIR / "{file}"),
)

jsoc_files

## 4. Optional: JSOC Cutout Template

Cutouts are useful when you only need a smaller field of view. This keeps downloads smaller and makes JSOC worthwhile.

The exact coordinates depend on your target. The example below is intentionally commented out so the notebook does not accidentally submit a cutout request.

In [ ]:
# from astropy.coordinates import SkyCoord
#
# bottom_left = SkyCoord(-300 * u.arcsec, -300 * u.arcsec, frame="helioprojective")
# top_right = SkyCoord(300 * u.arcsec, 300 * u.arcsec, frame="helioprojective")
#
# jsoc_cutout_query = Fido.search(
#     a.Time(START_TIME, END_TIME),
#     a.jsoc.Series("aia.lev1_euv_12s"),
#     a.jsoc.Segment("image"),
#     a.Wavelength(WAVELENGTH),
#     a.Sample(SAMPLE_CADENCE),
#     a.jsoc.Notify(JSOC_NOTIFY_EMAIL),
#     a.jsoc.Cutout(bottom_left=bottom_left, top_right=top_right, tracking=True),
# )
#
# jsoc_cutout_query

## 5. Which Method Should I Use?

| Situation | Recommended method |
| --- | --- |
| Quick look, short time range, full disk | Normal Fido |
| First attempt for a new dataset | Normal Fido |
| Normal route fails or is missing files | JSOC |
| You need stable AIA archive queries | JSOC |
| You need cutouts | JSOC |
| You are downloading a large range | JSOC, but start with a short test window |

A good workflow is: test a short time range with the normal method, then switch to JSOC if you need more reliability or a precise archive export.